# Entrainement Transfer Learning sur Colab (GPU)\n\nNotebook autonome : telecharge le dataset Kaggle, entraine EfficientNetB0 + ResNet50 + InceptionV3 (fine-tuning 2 phases), evalue sur un test set fige et exporte les resultats.

In [ ]:
# === Setup : donnees Kaggle + GPU check ===
import tensorflow as tf
print("TF", tf.__version__, "| GPU:", tf.config.list_physical_devices("GPU"))

# 1) Deposer kaggle.json (jeton API Kaggle)
from google.colab import files
import os
if not os.path.exists("/root/.kaggle/kaggle.json"):
    print(">>> Selectionnez votre kaggle.json")
    up = files.upload()
    os.makedirs("/root/.kaggle", exist_ok=True)
    for name in up:
        os.replace(name, "/root/.kaggle/kaggle.json")
    os.chmod("/root/.kaggle/kaggle.json", 0o600)

# 2) Telecharger + decompresser le dataset (~800 Mo)
import pathlib
DATA = pathlib.Path("/content/data/COVID-19_Radiography_Dataset")
if not (DATA / "COVID" / "images").is_dir():
    !pip -q install kaggle
    !kaggle datasets download -d tawsifurrahman/covid19-radiography-database -p /content
    !unzip -q -o /content/covid19-radiography-database.zip -d /content/data
print("Donnees pretes :", (DATA / "COVID" / "images").is_dir(), "->", DATA)

In [ ]:
# === Pipeline + entrainement 2 phases + evaluation (autonome) ===
import json, time, gc, numpy as np, pandas as pd, tensorflow as tf
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.utils.class_weight import compute_class_weight
from sklearn.metrics import (f1_score, balanced_accuracy_score, recall_score,
                             classification_report, confusion_matrix, roc_auc_score,
                             average_precision_score)
from sklearn.preprocessing import label_binarize
from tensorflow.keras import layers, models, optimizers
from tensorflow.keras.applications import (EfficientNetB0, ResNet50, InceptionV3,
                                           efficientnet, resnet, inception_v3)
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, CSVLogger

AUTOTUNE = tf.data.AUTOTUNE
CLASS_NAMES = ["COVID", "Lung_Opacity", "Normal", "Viral Pneumonia"]
C2I = {c:i for i,c in enumerate(CLASS_NAMES)}
SEED = 42
DATA = Path("/content/data/COVID-19_Radiography_Dataset")
OUT = Path("/content/results"); OUT.mkdir(exist_ok=True)

# --- Split stratifie fige 70/15/15 sur les chemins (anti-fuite) ---
rows = []
for c in CLASS_NAMES:
    for p in sorted((DATA/c/"images").glob("*.png")):
        rows.append((str(p), C2I[c]))
df = pd.DataFrame(rows, columns=["fp","label"])
tr, rest = train_test_split(df, test_size=0.30, stratify=df.label, random_state=SEED)
va, te = train_test_split(rest, test_size=0.5, stratify=rest.label, random_state=SEED)
print("split:", len(tr), len(va), len(te))
cw = compute_class_weight("balanced", classes=np.arange(4), y=tr.label.values)
class_weights = {i: float(w) for i,w in enumerate(cw)}
print("class_weights:", class_weights)

AUG = tf.keras.Sequential([layers.RandomRotation(0.03), layers.RandomZoom(0.10),
                           layers.RandomTranslation(0.05,0.05), layers.RandomFlip("horizontal")])

def make_ds(frame, img_size, prep, bs, training=False, tag="x"):
    fps = frame.fp.values; ys = frame.label.values
    ds = tf.data.Dataset.from_tensor_slices((fps, ys))
    def load(fp, y):
        img = tf.image.decode_png(tf.io.read_file(fp), channels=1)
        img = tf.image.grayscale_to_rgb(img)
        img = tf.image.resize(img, [img_size, img_size])
        return tf.cast(img, tf.float32), tf.one_hot(y, 4)
    ds = ds.map(load, num_parallel_calls=AUTOTUNE)
    ds = ds.cache(f"/content/cache_{tag}_{img_size}")  # cache images decodees -> epoques 2+ rapides
    if training: ds = ds.shuffle(4096, seed=SEED)
    ds = ds.batch(bs)
    ds = ds.map(lambda x,y:(prep(x),y), num_parallel_calls=AUTOTUNE)
    return ds.prefetch(AUTOTUNE)

SPECS = {
 "efficientnetb0": (224, efficientnet.preprocess_input, EfficientNetB0, ("block6","block7","top")),
 "resnet50":       (224, resnet.preprocess_input,       ResNet50,       ("conv5",)),
 "inceptionv3":    (299, inception_v3.preprocess_input, InceptionV3,    ("mixed8","mixed9","mixed10")),
}

def build(net, img_size, ctor):
    base = ctor(include_top=False, weights="imagenet", input_shape=(img_size,img_size,3))
    base.trainable = False
    inp = layers.Input((img_size,img_size,3))
    x = AUG(inp)                       # augmentation sur GPU (active en train uniquement)
    x = base(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.Dropout(0.3)(x)
    x = layers.Dense(256, activation="relu")(x)
    x = layers.Dropout(0.3)(x)
    out = layers.Dense(4, activation="softmax", dtype="float32")(x)
    return models.Model(inp, out), base

def freeze_bn(m):
    for l in getattr(m,"layers",[]):
        if isinstance(l, layers.BatchNormalization): l.trainable=False
        elif hasattr(l,"layers"): freeze_bn(l)

def evaluate(name, model, te_ds, y_true):
    proba = model.predict(te_ds, verbose=0); y_pred = proba.argmax(1)
    yb = label_binarize(y_true, classes=[0,1,2,3])
    rep = classification_report(y_true, y_pred, target_names=CLASS_NAMES, output_dict=True, zero_division=0)
    m = {"model":name,
         "f1_macro":float(f1_score(y_true,y_pred,average="macro")),
         "f1_weighted":float(f1_score(y_true,y_pred,average="weighted")),
         "balanced_accuracy":float(balanced_accuracy_score(y_true,y_pred)),
         "roc_auc_ovr_macro":float(roc_auc_score(yb,proba,average="macro",multi_class="ovr")),
         "pr_auc_ovr_macro":float(average_precision_score(yb,proba,average="macro")),
         "per_class":{c:{"precision":rep[c]["precision"],"recall":rep[c]["recall"],
                         "f1":rep[c]["f1-score"],"support":rep[c]["support"]} for c in CLASS_NAMES},
         "confusion_matrix":confusion_matrix(y_true,y_pred).tolist()}
    return m

def run(name, epochs1=6, epochs2=6):
    img_size, prep, ctor, ft = SPECS[name]
    bs = 16  # RAM safe
    tr_ds = make_ds(tr, img_size, prep, bs, training=True, tag=f"{name}_tr")
    va_ds = make_ds(va, img_size, prep, bs, tag=f"{name}_va")
    te_ds = make_ds(te, img_size, prep, bs, tag=f"{name}_te")
    model, base = build(name, img_size, ctor)
    print(f"\n### {name}: phase 1 (gele)"); t=time.time()
    model.compile(optimizers.Adam(1e-3), "categorical_crossentropy", metrics=["accuracy"])
    model.fit(tr_ds, validation_data=va_ds, epochs=epochs1, class_weight=class_weights,
              callbacks=[EarlyStopping(patience=3, restore_best_weights=True),
                         ReduceLROnPlateau(factor=0.2, patience=2),
                         CSVLogger(str(OUT/f"{name}_phase1.csv"))], verbose=2)
    base.trainable = True
    for l in base.layers: l.trainable = any(l.name.startswith(p) for p in ft)
    freeze_bn(base)
    print(f"### {name}: phase 2 (fine-tuning)")
    model.compile(optimizers.Adam(1e-5), "categorical_crossentropy", metrics=["accuracy"])
    model.fit(tr_ds, validation_data=va_ds, epochs=epochs2, class_weight=class_weights,
              callbacks=[EarlyStopping(patience=3, restore_best_weights=True),
                         ReduceLROnPlateau(factor=0.2, patience=2),
                         CSVLogger(str(OUT/f"{name}_phase2.csv"))], verbose=2)
    model.save(str(OUT/f"{name}_best.keras"))
    m = evaluate(name, model, te_ds, te.label.values)
    json.dump(m, open(OUT/f"{name}_test_metrics.json","w"), indent=2, ensure_ascii=False)
    print(f"[{name}] f1_macro={m['f1_macro']:.4f} bal_acc={m['balanced_accuracy']:.4f} "
          f"recall_COVID={m['per_class']['COVID']['recall']:.3f}  ({(time.time()-t)/60:.1f} min)", flush=True)
    del model; tf.keras.backend.clear_session(); gc.collect()
    return m

results = {}
for name in ["efficientnetb0","resnet50"]:
    try: results[name] = run(name)
    except Exception as e: print("ECHEC", name, e)

print("\n===RESULTS_JSON_START===")
print(json.dumps({k:{"f1_macro":v["f1_macro"],"balanced_accuracy":v["balanced_accuracy"],
                     "f1_weighted":v["f1_weighted"],"roc_auc":v["roc_auc_ovr_macro"],
                     "pr_auc":v["pr_auc_ovr_macro"],
                     "recall_COVID":v["per_class"]["COVID"]["recall"],
                     "recall_Viral":v["per_class"]["Viral Pneumonia"]["recall"]}
                  for k,v in results.items()}, indent=2))
print("===RESULTS_JSON_END===")

In [ ]:
# === Telecharger tous les resultats (JSON, CSV historiques, modeles) ===
import shutil
from google.colab import files
shutil.make_archive("/content/tl_results", "zip", "/content/results")
files.download("/content/tl_results.zip")